# Appendix Notebook A3: Single-Asset VaR and CVaR Estimation

This appendix notebook is the standardised review-paper version of the original local working file for Module 3.

This notebook supports the single-asset risk estimation section in Module 3. It documents the classical baseline, the quantum-inspired tail estimation logic, and the interpretation of loss-distribution outputs.

## Reading Guide

The workflow progresses from data ingestion to return construction, tail quantile estimation, stress interpretation, and scenario commentary.

All file names in `review_paper/notebooks/` follow a common naming convention so the paper appendix can reference them consistently.

# Module 3.6.1  VaR & CVaR of a Single-Asset Portfolio  
### Classical Baseline vs. Quantum-Inspired Tail Estimation

This notebook implements a complete demonstration of **Value-at-Risk (VaR)** and  
**Conditional Value-at-Risk (CVaR)** estimation for a single asset using:

---

## **1. Classical Methods**
- **Historical VaR / CVaR:**  
  Directly computed from empirical loss distribution.

- **Parametric (Normal) VaR / CVaR:**  
  Based on the estimated mean and standard deviation of daily log-returns.

These represent the standard techniques used in financial risk management systems.

---

## **2. Quantum-Inspired Tail Estimation**
To illustrate how quantum risk estimation ideas map into classical prototypes,  
we implement a **histogram-based discretisation** of the loss distribution:

- Losses are discretised into `N_BINS` intervals.  
- Each bin corresponds to a hypothetical quantum basis state \(|i\rangle\) with probability \(p_i\).  
- A "threshold oracle" is emulated by checking whether losses exceed a candidate threshold.  
- Tail probabilities and CVaR values are computed by scanning candidate thresholds.  

This mimics how **quantum amplitude estimation** or **quantum threshold oracles** would mark high-loss states,  
and how conditional sampling over the tail region would be used to estimate CVaR.

---

## **3. Implementation Details**
- Data is read from a bundled **Yahoo Finance** CSV snapshot covering 1 January 2024 to 31 December 2025.  
- Default ticker: **AAPL**, but any asset can be substituted.  
- We use the past 1 year of daily data by default (`LOOKBACK_DAYS = 365`).  
- The notebook outputs:
  - Historical VaR & CVaR  
  - Parametric (Normal) VaR & CVaR  
  - Quantum-Inspired VaR & CVaR  
  - A complete comparison table

---

## **4. Why This Case Is Important for Module 3**
This example serves as the simplest prototype of quantum-based risk estimation:

- Shows how a **loss distribution can be discretised** into quantum-friendly structures.  
- Demonstrates a **threshold-based oracle** that mirrors quantum amplitude filtering.  
- Provides an extendable foundation for:
  - multi-asset VaR/CVaR (Section 3.6.2),  
  - scenario propagation (Section 3.6.3),  
  - and quantum-enhanced tail sampling (Module 3.2).

---

## **Run the Code**
The next cell contains the full Python implementationready to run on any local machine.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm
from pathlib import Path

def locate_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'review_paper' / 'assets').exists():
            return candidate
    raise FileNotFoundError('Could not locate the repository root from the current working directory.')


def load_local_close_series(csv_path: Path, ticker: str) -> pd.Series:
    raw = pd.read_csv(csv_path, header=[0, 1])
    dates = pd.to_datetime(raw.iloc[1:, 0])
    closes = pd.to_numeric(raw.iloc[1:, 1], errors='coerce')
    return pd.Series(closes.to_numpy(), index=dates, name=ticker).dropna()

ROOT = locate_repo_root(Path.cwd())
ASSET_DATA_DIR = ROOT / 'review_paper' / 'assets' / 'module_03_risk' / 'data'
FIG_PATH = ROOT / 'review_paper' / 'single_asset_loss_plot.pdf'
ASSET_DATA_DIR.mkdir(parents=True, exist_ok=True)

import warnings
warnings.filterwarnings('ignore')


##### ==============================
##### 0. Parameter settings
##### ==============================

In [ ]:
TICKER = 'AAPL'        # Single asset ticker (e.g., MSFT, SPY, etc.)
ALPHA  = 0.95          # Confidence level for VaR / CVaR
HORIZON_DAYS = 1       # We compute 1-day VaR here
START_DATE = '2024-01-01'
END_DATE = '2025-12-31'


##### ==============================
##### 1. Download price data & compute returns
##### ==============================

In [ ]:
raw_csv_path = ASSET_DATA_DIR / f'{TICKER}_{START_DATE}_{END_DATE}_daily.csv'
if not raw_csv_path.exists():
    raise FileNotFoundError(f'Missing expected local CSV: {raw_csv_path}')

print(f'Loading {TICKER} from {START_DATE} to {END_DATE} (inclusive) ...')
prices = load_local_close_series(raw_csv_path, TICKER)

# Daily log returns
rets = np.log(prices / prices.shift(1)).dropna()

# Define 'loss' as -return (positive means a loss)
losses = -rets.values
n = len(losses)
print(f'Sample size (daily returns): {n}')
print(f'Using local market data from: {raw_csv_path}')


##### ==============================
##### 2. Classical baseline: VaR & CVaR
##### ==============================

In [ ]:
# 2.1 Historical VaR / CVaR
# VaR_alpha: P(loss <= VaR_alpha) = alpha, i.e., alpha-quantile of loss distribution
var_hist = np.quantile(losses, ALPHA)

# CVaR_alpha: Conditional expectation E[ loss | loss >= VaR_alpha ]
tail_losses = losses[losses >= var_hist]
cvar_hist = tail_losses.mean()

# 2.2 Parametric (normal) VaR / CVaR
mu_hat    = losses.mean()
sigma_hat = losses.std(ddof=1)

# For a normal distribution N(mu, sigma^2):
# VaR_alpha = mu + sigma * Phi^{-1}(alpha)
z_alpha   = norm.ppf(ALPHA)
var_norm  = mu_hat + sigma_hat * z_alpha

# Closed-form CVaR for normal losses:
# CVaR_alpha = mu + sigma * (z_alpha) / (1 - alpha)
phi_alpha   = norm.pdf(z_alpha)
cvar_norm   = mu_hat + sigma_hat * (phi_alpha / (1 - ALPHA))

print("\n=== Classical Baseline (1-day, single asset) ===")
print(f"Alpha = {ALPHA:.2%}")
print(f"[Historical]     VaR  = {var_hist:.4%},   CVaR = {cvar_hist:.4%}")
print(f"[Normal (param)] VaR  = {var_norm:.4%},   CVaR = {cvar_norm:.4%}")

##### ==============================
##### 3. Quantum-inspired tail estimation
##### ==============================

In [ ]:
# Idea: discretise the loss distribution into bins.
# Each bin corresponds to a basis state |i> with probability p_i.
# A quantum-style threshold oracle marks losses above threshold L*.
# Conditional sampling on the marked region gives tail expectation.

N_BINS = 64  # Number of bins ( 6 qubits), can be increased if desired

# 3.1 Histogram discretisation to obtain bins & probability vector p
hist_counts, bin_edges = np.histogram(losses, bins=N_BINS, density=False)
p = hist_counts / hist_counts.sum()  # Probability mass function

bin_centers = 0.5 * (bin_edges[1:] + bin_edges[:-1])

# 3.2 Convert threshold oracle + tail probability into a grid search
#     For each candidate threshold L_candidate:
#     P(loss >= L_candidate) = sum_{i: bin_centers[i] >= L_candidate} p_i
#     Choose L* whose tail probability best matches (1 - ALPHA)

target_tail_prob = 1.0 - ALPHA

# Filter across all possible thresholds (bin centers)
candidate_VaRs = bin_centers

tail_probs = []
for L_cand in candidate_VaRs:
    mask = (bin_centers >= L_cand)
    tail_prob = p[mask].sum()
    tail_probs.append(tail_prob)

tail_probs = np.array(tail_probs)

# Identify the threshold whose tail probability is closest to target
idx_best = np.argmin(np.abs(tail_probs - target_tail_prob))
var_qi   = candidate_VaRs[idx_best]
tail_prob_best = tail_probs[idx_best]

# 3.3 Quantum-inspired CVaR:
#     Compute conditional expectation over the tail region
mask_tail_bins = (bin_centers >= var_qi)
p_tail = p[mask_tail_bins]
L_tail = bin_centers[mask_tail_bins]

# Safety check to avoid division by zero
if p_tail.sum() > 0:
    cvar_qi = (p_tail * L_tail).sum() / p_tail.sum()
else:
    cvar_qi = var_qi  # Fallback in degenerate case

print("\n=== Quantum-inspired (histogram + threshold search) ===")
print(f"Target tail prob: 1 - alpha = {target_tail_prob:.2%}")
print(f"Chosen VaR (grid)   = {var_qi:.4%}")
print(f"Tail prob at VaR    = {tail_prob_best:.2%}")
print(f"CVaR (grid tail mean)= {cvar_qi:.4%}")

##### ==============================
##### 4. Summary comparison
##### ==============================

In [ ]:
print("\n=== Summary (all in 1-day loss %) ===")
print(f"Historical VaR  : {var_hist*100:.3f}%,   CVaR: {cvar_hist*100:.3f}%")
print(f"Normal VaR      : {var_norm*100:.3f}%,   CVaR: {cvar_norm*100:.3f}%")
print(f'QI (hist-grid) V: {var_qi*100:.3f}%,   CVaR: {cvar_qi*100:.3f}%')

##### ==============================
##### 5. Plot loss histogram + VaR/CVaR lines
##### ==============================

In [ ]:
plt.figure(figsize=(10, 6))

# Histogram of losses
plt.hist(losses, bins=50, density=True, alpha=0.6,
         color='steelblue', label='Loss Distribution')

# Historical VaR / CVaR
plt.axvline(var_hist, color='red', linestyle='--', linewidth=2,
            label=f'Historical VaR ({var_hist*100:.2f}%)')
plt.axvline(cvar_hist, color='red', linestyle=':', linewidth=2,
            label=f'Historical CVaR ({cvar_hist*100:.2f}%)')

# Normal-parametric VaR / CVaR
plt.axvline(var_norm, color='blue', linestyle='--', linewidth=2,
            label=f'Normal VaR ({var_norm*100:.2f}%)')
plt.axvline(cvar_norm, color='blue', linestyle=':', linewidth=2,
            label=f'Normal CVaR ({cvar_norm*100:.2f}%)')

# Quantum-inspired VaR / CVaR
plt.axvline(var_qi, color='purple', linestyle='--', linewidth=2,
            label=f'QI VaR ({var_qi*100:.2f}%)')
plt.axvline(cvar_qi, color='purple', linestyle=':', linewidth=2,
            label=f'QI CVaR ({cvar_qi*100:.2f}%)')

# Plot Formatting
plt.title(f'Loss Distribution with VaR/CVaR  {TICKER} (1-day horizon)')
plt.xlabel('Loss')
plt.ylabel('Density')
plt.grid(alpha=0.25)
plt.legend()

out_pdf = FIG_PATH
plt.savefig(out_pdf, format='pdf', bbox_inches='tight')
print('Saved:', out_pdf)

plt.show()
plt.close()


## Interpretation Note

This notebook is preserved in the appendix asset set so the review paper can keep executable detail separate from the main argument.

In [ ]:
summary = pd.DataFrame([
    {
        'method': 'Historical',
        'var_1d_pct': var_hist * 100,
        'cvar_1d_pct': cvar_hist * 100,
        'notes': 'empirical quantile',
    },
    {
        'method': 'Normal-parametric',
        'var_1d_pct': var_norm * 100,
        'cvar_1d_pct': cvar_norm * 100,
        'notes': 'fitted normal loss model',
    },
    {
        'method': 'Quantum-inspired grid',
        'var_1d_pct': var_qi * 100,
        'cvar_1d_pct': cvar_qi * 100,
        'notes': 'histogram-based threshold oracle',
    },
])
summary_path = ASSET_DATA_DIR / 'module_03_single_asset_var_cvar_summary.csv'
summary.to_csv(summary_path, index=False, float_format='%.3f')
display(summary)
print(f'Summary saved to: {summary_path}')
